In [26]:
from crewai.rag.embeddings.providers.ollama.types import OllamaProviderSpec

embedding_model: OllamaProviderSpec = {
    "provider": "ollama",
    "config": {
        "model_name": "nomic-embed-text",
        "url": "http://localhost:11434/api/embeddings"
    }
}

In [27]:
from crewai_tools import RagTool
from crewai import Agent, Crew, Task, Process
from crewai.project import agent, crew, task, CrewBase





In [28]:
# Create a RAG tool with default settings
rag_tool = RagTool()

In [ ]:
# Add a PDF file
rag_tool.add(data_type="file", path="knowledge/NY-Wind-Energy-Guide-1.pdf")

# Add a web page
rag_tool.add(data_type="website", url="https://www.nationalgrid.com/stories/energy-explained/how-does-solar-power-work/")
rag_tool.add(data_type="website", url="https://education.nationalgeographic.org/resource/solar-energy/")

# Add a YouTube video
rag_tool.add(data_type="youtube_video", url="https://www.youtube.com/watch?v=3dvEx5XvxUc")

rag_tool.add(data_type="directory", path="knowledge/nuclear")

In [29]:
import asyncio

In [ ]:
from crewai import Agent, LLM

llm = LLM(
    model="ollama/qwen2.5:3b",
    base_url="http://localhost:11434",
)

In [30]:
knowledge_expert = Agent(
    role="Energy Knowledge Expert",

    goal=(
        "Answer questions about energy technologies using ONLY "
        "information retrieved from the provided energy knowledge base."
    ),

    backstory=(
        "You are an energy-domain knowledge retrieval expert. "
        "Your knowledge is strictly limited to the information "
        "contained in the provided RAG knowledge base. "
        "The knowledge base contains information about technologies "
        "such as solar energy, wind energy, biogas, nuclear energy, "
        "hydropower, and other explicitly provided energy technologies."
    ),

    tools=[rag_tool],
    llm="gemini/gemini-3.8-flash",
    allow_delegation=False,
    verbose=True,
)

In [31]:
knowledge_task = Task(
    description="""
    Answer the following question:

    {question}

    IMPORTANT RULES:

    1. You MUST use the provided RAG tool to retrieve information
       relevant to the question.

    2. DO NOT use your pretrained/general knowledge to fill gaps.

    3. Try your best to understand the question and answer it using the given information.

    4. If and only if the required information cannot be found in the
       knowledge base, explicitly say:

       "The provided knowledge base does not contain enough
       information to answer this question."

   
    6. If only part of the question can be answered from the
       knowledge base, answer only that part and clearly state
       which information is unavailable.

    7. Do not retrieve information from the internet or any
       external knowledge source.

    8. Prefer information directly supported by the retrieved
       documents over assumptions or general reasoning.
    """,

    expected_output=(
        "A factual answer based exclusively on information retrieved "
        "from the provided energy knowledge base. If the knowledge "
        "base does not contain sufficient information, explicitly "
        "state that the information is unavailable."
    ),

    agent=knowledge_expert,
)


In [32]:

energy_crew = Crew(
    agents=[knowledge_expert],
    tasks=[knowledge_task],
    process=Process.sequential,
    verbose=True,
)


In [33]:
async def answer_question(question: str):

    return await energy_crew.kickoff_async(
        inputs={"question": question}
    )

In [ ]:
res = await answer_question(
    "Which countries are leading in the development of nuclear energy technologies, and what are their key contributions?"
)

print(res)

In [34]:
res = await answer_question(
    "List some countries using nuclear energy technologies, and how they use them."
)

print(res)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.22                                                                                       │
│  Latest version:  1.15.23                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d6703512-cc79-4ca1-9d43-00015e007d48                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Answer the following question:                                                                             │
│                                                                                                                 │
│      List some countries using nuclear energy technologies, and how they use them.                              │
│                                                                                                                 │
│      IMPORTANT RULES:                                                                                           │
│                                                                                                                 │
│      1. You MUST use the provided RAG tool to retrieve information                                              │
│         relevant to the question.                                                                               │
│                                                                                                                 │
│      2. DO NOT use your pretrained/general knowledge to fill gaps.                                              │
│                                                                                                                 │
│      3. Try your best to understand the question and answer it using the given information.                     │
│                                                                                                                 │
│      4. If and only if the required information cannot be found in the                                          │
│         knowledge base, explicitly say:                                                                         │
│                                                                                                                 │
│         "The provided knowledge base does not contain enough                                                    │
│         information to answer this question."                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│      6. If only part of the question can be answered from the                                                   │
│         knowledge base, answer only that part and clearly state                                                 │
│         which information is unavailable.                                                                       │
│                                                                                                                 │
│      7. Do not retrieve information from the internet or any                                                    │
│         external knowledge source.                                                                              │
│                                                                                                                 │
│      8. Prefer information directly supported by the retrieved                                                  │
│         documents over assumptions or general reasoning.                                                        │
│                                                                                                                 │
│  ID: 4139532b-76d4-46ea-a819-f28b0816c401              

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Energy Knowledge Expert                                                                                 │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Answer the following question:                                                                             │
│                                                                                                                 │
│      List some countries using nuclear energy technologies, and how they use them.                              │
│                                                                                                                 │
│      IMPORTANT RULES:                                                                                           │
│                                                                                                                 │
│      1. You MUST use the provided RAG tool to retrieve information                                              │
│         relevant to the question.                                                                               │
│                                                                                                                 │
│      2. DO NOT use your pretrained/general knowledge to fill gaps.                                              │
│                                                                                                                 │
│      3. Try your best to understand the question and answer it using the given information.                     │
│                                                                                                                 │
│      4. If and only if the required information cannot be found in the                                          │
│         knowledge base, explicitly say:                                                                         │
│                                                                                                                 │
│         "The provided knowledge base does not contain enough                                                    │
│         information to answer this question."                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│      6. If only part of the question can be answered from the                                                   │
│         knowledge base, answer only that part and clearly state                                                 │
│         which information is unavailable.                                                                       │
│                                                                                                                 │
│      7. Do not retrieve information from the internet or any                                                    │
│         external knowledge source.                                                                              │
│                                                                                                                 │
│      8. Prefer information directly supported by the retrieved                                                  │
│         documents over assumptions or general reasoning.                                                        │
│                                                        

╭──────────────────────────────────────── 🔧 Tool Execution Started (#5) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: knowledge_base                                                                                           │
│  Args: {'limit': 10, 'query': 'countries using nuclear energy technologies', 'similarity_threshold': 0.5}       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Energy Knowledge Expert                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The provided knowledge base does not contain enough information to answer this question.                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Answer the following question:                                                                             │
│                                                                                                                 │
│      List some countries using nuclear energy technologies, and how they use them.                              │
│                                                                                                                 │
│      IMPORTANT RULES:                                                                                           │
│                                                                                                                 │
│      1. You MUST use the provided RAG tool to retrieve information                                              │
│         relevant to the question.                                                                               │
│                                                                                                                 │
│      2. DO NOT use your pretrained/general knowledge to fill gaps.                                              │
│                                                                                                                 │
│      3. Try your best to understand the question and answer it using the given information.                     │
│                                                                                                                 │
│      4. If and only if the required information cannot be found in the                                          │
│         knowledge base, explicitly say:                                                                         │
│                                                                                                                 │
│         "The provided knowledge base does not contain enough                                                    │
│         information to answer this question."                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│      6. If only part of the question can be answered from the                                                   │
│         knowledge base, answer only that part and clearly state                                                 │
│         which information is unavailable.                                                                       │
│                                                                                                                 │
│      7. Do not retrieve information from the internet or any                                                    │
│         external knowledge source.                                                                              │
│                                                                                                                 │
│      8. Prefer information directly supported by the retrieved                                                  │
│         documents over assumptions or general reasoning.                                                        │
│                                                                                                                 │
│  Agent: Energy Knowledge Expert                        

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d6703512-cc79-4ca1-9d43-00015e007d48                                                                       │
│  Final Output: The provided knowledge base does not contain enough information to answer this question.         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The provided knowledge base does not contain enough information to answer this question.


In [35]:
res = await answer_question(
    "What is nuclear energy?"
)

print(res)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.22                                                                                       │
│  Latest version:  1.15.23                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d6703512-cc79-4ca1-9d43-00015e007d48                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Answer the following question:                                                                             │
│                                                                                                                 │
│      What is nuclear energy?                                                                                    │
│                                                                                                                 │
│      IMPORTANT RULES:                                                                                           │
│                                                                                                                 │
│      1. You MUST use the provided RAG tool to retrieve information                                              │
│         relevant to the question.                                                                               │
│                                                                                                                 │
│      2. DO NOT use your pretrained/general knowledge to fill gaps.                                              │
│                                                                                                                 │
│      3. Try your best to understand the question and answer it using the given information.                     │
│                                                                                                                 │
│      4. If and only if the required information cannot be found in the                                          │
│         knowledge base, explicitly say:                                                                         │
│                                                                                                                 │
│         "The provided knowledge base does not contain enough                                                    │
│         information to answer this question."                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│      6. If only part of the question can be answered from the                                                   │
│         knowledge base, answer only that part and clearly state                                                 │
│         which information is unavailable.                                                                       │
│                                                                                                                 │
│      7. Do not retrieve information from the internet or any                                                    │
│         external knowledge source.                                                                              │
│                                                                                                                 │
│      8. Prefer information directly supported by the retrieved                                                  │
│         documents over assumptions or general reasoning.                                                        │
│                                                                                                                 │
│  ID: 4139532b-76d4-46ea-a819-f28b0816c401              

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Energy Knowledge Expert                                                                                 │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Answer the following question:                                                                             │
│                                                                                                                 │
│      What is nuclear energy?                                                                                    │
│                                                                                                                 │
│      IMPORTANT RULES:                                                                                           │
│                                                                                                                 │
│      1. You MUST use the provided RAG tool to retrieve information                                              │
│         relevant to the question.                                                                               │
│                                                                                                                 │
│      2. DO NOT use your pretrained/general knowledge to fill gaps.                                              │
│                                                                                                                 │
│      3. Try your best to understand the question and answer it using the given information.                     │
│                                                                                                                 │
│      4. If and only if the required information cannot be found in the                                          │
│         knowledge base, explicitly say:                                                                         │
│                                                                                                                 │
│         "The provided knowledge base does not contain enough                                                    │
│         information to answer this question."                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│      6. If only part of the question can be answered from the                                                   │
│         knowledge base, answer only that part and clearly state                                                 │
│         which information is unavailable.                                                                       │
│                                                                                                                 │
│      7. Do not retrieve information from the internet or any                                                    │
│         external knowledge source.                                                                              │
│                                                                                                                 │
│      8. Prefer information directly supported by the retrieved                                                  │
│         documents over assumptions or general reasoning.                                                        │
│                                                        

╭──────────────────────────────────────── 🔧 Tool Execution Started (#9) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: knowledge_base                                                                                           │
│  Args: {'limit': 5, 'query': 'What is nuclear energy?', 'similarity_threshold': 0.5}                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Energy Knowledge Expert                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The provided knowledge base does not contain a dedicated definition or comprehensive information explaining    │
│  what **nuclear energy** is as an energy technology or power source (such as nuclear fission or nuclear power   │
│  generation).                                                                                                   │
│                                                                                                                 │
│  The only related information available in the knowledge base concerns **nuclear fusion**:                      │
│                                                                                                                 │
│  * **Nuclear fusion definition:** A "process where the nuclei of one element, usually hydrogen, fuse with each  │
│  other to form the nuclei of another element, usually helium."                                                  │
│  * **Process and energy release:**                                                                              │
│    * "Solar energy is created by nuclear fusion that takes place in the sun."                                   │
│    * "Fusion occurs when protons of hydrogen atoms violently collide in the sun’s core and fuse to create a     │
│  helium atom. This process, known as a PP (proton-proton) chain reaction, emits an enormous amount of energy."  │
│    * "Nuclear fusion by the PP chain reaction or CNO cycle releases tremendous amounts of energy in the form    │
│  of waves and particles."                                                                                       │
│                                                                                                                 │
│  Any broader explanation or general details regarding nuclear energy are unavailable. Therefore:                │
│                                                                                                                 │
│  "The provided knowledge base does not contain enough information to answer this question."                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Answer the following question:                                                                             │
│                                                                                                                 │
│      What is nuclear energy?                                                                                    │
│                                                                                                                 │
│      IMPORTANT RULES:                                                                                           │
│                                                                                                                 │
│      1. You MUST use the provided RAG tool to retrieve information                                              │
│         relevant to the question.                                                                               │
│                                                                                                                 │
│      2. DO NOT use your pretrained/general knowledge to fill gaps.                                              │
│                                                                                                                 │
│      3. Try your best to understand the question and answer it using the given information.                     │
│                                                                                                                 │
│      4. If and only if the required information cannot be found in the                                          │
│         knowledge base, explicitly say:                                                                         │
│                                                                                                                 │
│         "The provided knowledge base does not contain enough                                                    │
│         information to answer this question."                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│      6. If only part of the question can be answered from the                                                   │
│         knowledge base, answer only that part and clearly state                                                 │
│         which information is unavailable.                                                                       │
│                                                                                                                 │
│      7. Do not retrieve information from the internet or any                                                    │
│         external knowledge source.                                                                              │
│                                                                                                                 │
│      8. Prefer information directly supported by the retrieved                                                  │
│         documents over assumptions or general reasoning.                                                        │
│                                                                                                                 │
│  Agent: Energy Knowledge Expert                        

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d6703512-cc79-4ca1-9d43-00015e007d48                                                                       │
│  Final Output: The provided knowledge base does not contain a dedicated definition or comprehensive             │
│  information explaining what **nuclear energy** is as an energy technology or power source (such as nuclear     │
│  fission or nuclear power generation).                                                                          │
│                                                                                                                 │
│  The only related information available in the knowledge base concerns **nuclear fusion**:                      │
│                                                                                                                 │
│  * **Nuclear fusion definition:** A "process where the nuclei of one element, usually hydrogen, fuse with each  │
│  other to form the nuclei of another element, usually helium."                                                  │
│  * **Process and energy release:**                                                                              │
│    * "Solar energy is created by nuclear fusion that takes place in the sun."                                   │
│    * "Fusion occurs when protons of hydrogen atoms violently collide in the sun’s core and fuse to create a     │
│  helium atom. This process, known as a PP (proton-proton) chain reaction, emits an enormous amount of energy."  │
│    * "Nuclear fusion by the PP chain reaction or CNO cycle releases tremendous amounts of energy in the form    │
│  of waves and particles."                                                                                       │
│                                                                                                                 │
│  Any broader explanation or general details regarding nuclear energy are unavailable. Therefore:                │
│                                                                                                                 │
│  "The provided knowledge base does not contain enough information to answer this question."                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The provided knowledge base does not contain a dedicated definition or comprehensive information explaining what **nuclear energy** is as an energy technology or power source (such as nuclear fission or nuclear power generation). 

The only related information available in the knowledge base concerns **nuclear fusion**:

* **Nuclear fusion definition:** A "process where the nuclei of one element, usually hydrogen, fuse with each other to form the nuclei of another element, usually helium."
* **Process and energy release:**
  * "Solar energy is created by nuclear fusion that takes place in the sun."
  * "Fusion occurs when protons of hydrogen atoms violently collide in the sun’s core and fuse to create a helium atom. This process, known as a PP (proton-proton) chain reaction, emits an enormous amount of energy."
  * "Nuclear fusion by the PP chain reaction or CNO cycle releases tremendous amounts of energy in the form of waves and particles."

Any broader explanation or general details 